<a href="https://colab.research.google.com/github/tanya-quiroz/json-diffs/blob/main/Tokens_Merger_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import time
from datetime import datetime
from google.colab import files

def smart_merge():
    # 1. UPLOAD FILES
    print("👇 Please upload exactly TWO files (your Engineer JSON and your Designer JSON):")
    uploaded = files.upload()
    uploaded_filenames = list(uploaded.keys())

    # Check that exactly two files were uploaded
    if len(uploaded_filenames) != 2:
        print(f"\n❌ Error: You uploaded {len(uploaded_filenames)} files. Please run the script again and upload exactly 2 files.")
        return

    # 2. IDENTIFY FILES
    # Look for the specific Engineer naming pattern
    engineer_filename = next((f for f in uploaded_filenames if f.startswith("Figma Tokens - Motif.")), None)

    if not engineer_filename:
        print("\n❌ Error: Could not identify the Engineer's file.")
        print("Make sure the Engineer's file starts with 'Figma Tokens - Motif.'")
        return

    # The designer file is whichever one is NOT the engineer file
    designer_filename = next(f for f in uploaded_filenames if f != engineer_filename)

    print(f"\n✅ Identified Engineer file: {engineer_filename}")
    print(f"✅ Identified Designer file: {designer_filename}")
    print("\n...Reading and merging files...")

    # 3. LOAD CONTENT
    eng_data = json.loads(uploaded[engineer_filename].decode('utf-8'))
    figma_data = json.loads(uploaded[designer_filename].decode('utf-8'))

    total_changes = 0
    changelog_lines = ["--- Token Merge Changelog ---", f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ""]

    # Helper function to print and log at the same time
    def log_change(message):
        print(message)
        changelog_lines.append(message)

    # 4. RECURSIVE MERGE FUNCTION
    def recursive_update(target, source, path=""):
        count = 0

        # --- PHASE 1: DELETIONS ---
        keys_to_delete = []
        for key in target.keys():
            if key.startswith("$"): # Never delete metadata
                continue
            if key not in source:
                keys_to_delete.append(key)

        for key in keys_to_delete:
            log_change(f"  [-] Removed: {path} > {key}")
            del target[key]
            count += 1

        # --- PHASE 2: ADDITIONS & UPDATES ---
        for key, source_val in source.items():
            if key.startswith("$"): # Skip metadata in source
                continue

            # If key is missing in Target, add it (New Token)
            if key not in target:
                log_change(f"  [+] Added: {path} > {key}")
                target[key] = source_val
                count += 1
                continue

            target_val = target[key]

            # Check if both are Token Objects (have a 'value')
            if isinstance(source_val, dict) and 'value' in source_val:
                if isinstance(target_val, dict) and 'value' in target_val:

                    # Update Value if different
                    if source_val['value'] != target_val['value']:
                        log_change(f"  [~] Updated Value {path} > {key}: {target_val['value']} -> {source_val['value']}")
                        target_val['value'] = source_val['value']
                        count += 1

                    # Update Description ONLY if Engineer has one and it's NOT empty
                    source_desc = source_val.get('description', '').strip()
                    target_desc = target_val.get('description', '').strip()

                    if source_desc and source_desc != target_desc:
                        log_change(f"  [i] Updated Description {path} > {key}")
                        target_val['description'] = source_desc

            # If they are Groups (nested folders), dig deeper
            elif isinstance(source_val, dict) and isinstance(target_val, dict):
                count += recursive_update(target_val, source_val, path + " > " + key)

        return count

    # --- EXECUTION ---
    # First, handle top-level deletions
    sets_to_delete = []
    for token_set in figma_data.keys():
        if token_set.startswith("$"): continue
        if token_set not in eng_data:
            sets_to_delete.append(token_set)

    for token_set in sets_to_delete:
        log_change(f"\n[-] Removed entire Token Set: {token_set}")
        del figma_data[token_set]
        total_changes += 1

    # Second, iterate through the top-level sets to update and add
    for token_set in eng_data.keys():
        if token_set.startswith("$"): continue

        if token_set in figma_data:
            log_change(f"\nScanning Set: {token_set}...")
            total_changes += recursive_update(figma_data[token_set], eng_data[token_set], token_set)
        else:
            log_change(f"\n[+] Added entire Token Set: {token_set}")
            figma_data[token_set] = eng_data[token_set]
            total_changes += 1

    summary_msg = f"\n✅ Success! {total_changes} changes were made (additions, updates, and removals)."
    log_change(summary_msg)

    # 5. DYNAMIC NAMING & DOWNLOADING
    current_date = datetime.now().strftime("%Y%m%d") # Format: YYYYMMDD
    output_filename = f"merged_tokens_{current_date}.json"
    changelog_filename = f"merged_tokens_changelog_{current_date}.txt"

    # Save JSON
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(figma_data, f, indent=2)

    # Save Changelog
    with open(changelog_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(changelog_lines))

    print(f"\n⬇️ Downloading '{output_filename}' and '{changelog_filename}' now...")
    files.download(output_filename)
    time.sleep(1) # Tiny pause to ensure browser catches both downloads
    files.download(changelog_filename)

# Run the script
try:
    smart_merge()
except Exception as e:
    print(f"An error occurred: {e}")

👇 Please upload exactly TWO files (your Engineer JSON and your Designer JSON):


Saving Figma Tokens - Motif.2026-03-25T16_31_34.377Z.json to Figma Tokens - Motif.2026-03-25T16_31_34.377Z.json
Saving SDS-M-DesignTeam-tokens-20260325.json to SDS-M-DesignTeam-tokens-20260325.json

✅ Identified Engineer file: Figma Tokens - Motif.2026-03-25T16_31_34.377Z.json
✅ Identified Designer file: SDS-M-DesignTeam-tokens-20260325.json

...Reading and merging files...

Scanning Set: Snap Motif/Global...
  [-] Removed: Snap Motif/Global > Palette
  [~] Updated Value Snap Motif/Global > Root > --h1-font-family: Program Nar OT, Helvetica, Tahoma, Arial, sans-serif -> Program OT, Helvetica Heading, Tahoma Heading, Arial, sans-serif
  [~] Updated Value Snap Motif/Global > Root > --h2-font-family: Program Nar OT, Helvetica, Tahoma, Arial, sans-serif -> Program OT, Helvetica Heading, Tahoma Heading, Arial, sans-serif
  [+] Added: Snap Motif/Global > Palette.Plain
  [+] Added: Snap Motif/Global > Palette.Black
  [+] Added: Snap Motif/Global > Palette.Yellow
  [+] Added: Snap Motif/Global

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>